# Embeddings

The parsing notebook produced a flat `list[Chunk]` — plain text with metadata. Before we can retrieve anything, each chunk needs to become a vector so ChromaDB can find the nearest neighbours to a query.

This notebook:

1. Loads the parsed chunks
2. Embeds every chunk's `.text` with a local `sentence-transformers` model
3. Upserts the vectors + metadata into ChromaDB
4. Runs a test query end-to-end to verify the store works

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

# utils.py lives one level up — add the project root so we can import it.
sys.path.insert(0, str(Path("..").resolve()))
from utils import Chunk, get_chroma_client, add_chunks

# ── Embedding model ───────────────────────────────────────────────────────────
# all-MiniLM-L6-v2: 384-dimensional, ~80 MB, fast on CPU.
# Good baseline for retrieval — swap for a larger model if recall is poor.
EMBED_MODEL = "all-MiniLM-L6-v2"

# ── Batch size ────────────────────────────────────────────────────────────────
# Number of chunks to embed in one forward pass.
# 64 is safe on most hardware; raise to 128-256 if you have a GPU.
BATCH_SIZE = 64

# ── ChromaDB ──────────────────────────────────────────────────────────────────
CHROMA_DIR   = "../chroma_db"   # persistent storage directory
COLLECTION   = "finance_rag"    # collection name inside ChromaDB

print("Setup complete.")

## 2. Load Chunks

Load the chunks saved by `01_parsing_strategy.ipynb`. Run that notebook first if `chunks_cache.pkl` does not exist yet.

In [ ]:
import pickle

cache_path = Path("../chunks_cache.pkl")

if not cache_path.exists():
    raise FileNotFoundError(
        f"Cache not found at {cache_path}. "
        "Run 01_parsing_strategy.ipynb first to generate it."
    )

with open(cache_path, "rb") as f:
    all_chunks = pickle.load(f)

print(f"Loaded {len(all_chunks)} chunks from cache")

In [ ]:
# Sanity-check the loaded chunks before spending time on embeddings.
n_pdf  = sum(1 for c in all_chunks if c.source.endswith(".pdf"))
n_htm  = sum(1 for c in all_chunks if c.source.endswith(".htm"))
n_tbl  = sum(1 for c in all_chunks if c.content_type == "table")
n_prse = sum(1 for c in all_chunks if c.content_type == "prose")

print(f"Total chunks : {len(all_chunks)}")
print(f"  from PDFs  : {n_pdf}")
print(f"  from HTMs  : {n_htm}")
print(f"  tables     : {n_tbl}")
print(f"  prose      : {n_prse}")

## 3. Embed

We embed every chunk's `.text` in batches.

**Why batches?**  
Passing all chunks at once would load every text string into a single tensor, which can exhaust RAM for large corpora. Processing in fixed-size batches keeps memory flat regardless of corpus size.

In [ ]:
# Load the model once — this downloads ~80 MB on first run and caches it locally.
model = SentenceTransformer(EMBED_MODEL)
print(f"Model loaded: {EMBED_MODEL}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
texts = [c.text for c in all_chunks]

all_embeddings = []  # will hold one numpy array per chunk

for start in range(0, len(texts), BATCH_SIZE):
    batch = texts[start : start + BATCH_SIZE]

    # encode() returns a (batch_size, dim) numpy array.
    # convert_to_numpy=True ensures we always get numpy regardless of backend.
    vecs = model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
    all_embeddings.extend(vecs.tolist())

    # Print progress every 10 batches so long runs don't look frozen.
    if (start // BATCH_SIZE) % 10 == 0:
        print(f"  embedded {min(start + BATCH_SIZE, len(texts))}/{len(texts)} chunks")

print(f"\nDone. {len(all_embeddings)} embeddings, dim={len(all_embeddings[0])}")

## 4. Store in ChromaDB

`add_chunks` (from `utils.py`) calls `collection.upsert`, so re-running this cell is safe — existing chunks are overwritten with the same ID rather than duplicated.

In [ ]:
client     = get_chroma_client(CHROMA_DIR)
collection = client.get_or_create_collection(COLLECTION)

# Upsert in the same batches used for embedding so memory stays flat.
for start in range(0, len(all_chunks), BATCH_SIZE):
    batch_chunks = all_chunks[start : start + BATCH_SIZE]
    batch_vecs   = all_embeddings[start : start + BATCH_SIZE]
    add_chunks(collection, batch_chunks, batch_vecs)

    if (start // BATCH_SIZE) % 10 == 0:
        print(f"  upserted {min(start + BATCH_SIZE, len(all_chunks))}/{len(all_chunks)} chunks")

print(f"\nCollection '{COLLECTION}' now has {collection.count()} documents.")

## 5. Verify — Run a Test Query

Embed a query with the same model used for indexing, then ask ChromaDB for the five nearest chunks. If the top results are on-topic, the pipeline is working.

In [ ]:
QUERY = "What was Amazon's revenue in 2024?"

# Embed the query with the same model — vector space must match.
query_vec = model.encode(QUERY, convert_to_numpy=True).tolist()

results = collection.query(
    query_embeddings=[query_vec],
    n_results=5,
    include=["documents", "metadatas", "distances"],
)

print(f"Query: {QUERY}\n")
print(f"{'Rank':<5} {'Score':>6}  {'Source':<45}  {'Type':<6}  Preview")
print("-" * 110)

for rank, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
), start=1):
    # ChromaDB returns L2 distances — lower is better.
    # Convert to a 0-1 similarity score for readability.
    score = 1 / (1 + dist)
    source  = meta.get("source", "")[-45:]
    ct      = meta.get("content_type", "")[:6]
    preview = doc[:80].replace("\n", " ")
    print(f"{rank:<5} {score:>6.3f}  {source:<45}  {ct:<6}  {preview}...")